In [1]:
using XLSX, DataFrames, Glob, JSON, Random, StatsBase


In [2]:

function generate_remaining_time(aircraft_airport::Dict, n::Int)
    tail_numbers = collect(keys(aircraft_airport))
    @assert n <= length(tail_numbers) 
    
    # Sélection aléatoire de n avions parmi ceux fournis
    selected = sample(tail_numbers, n, replace=false)

    # Création du DataFrame résultat
    result_df = DataFrame(
        Aircraft       = tail_numbers,
        Airport        = [aircraft_airport[t] for t in tail_numbers],
        RemainingTime  = [t in selected ? rand(200:3000) : ">5740" for t in tail_numbers]
    )
     # Trier : valeurs numériques d'abord (croissant), ">5740" à la fin
    sort!(result_df, :RemainingTime, by = x -> x == ">5740" ? Inf : x)

    println("Avions sélectionnés : ", selected)
    return result_df
end

generate_remaining_time (generic function with 1 method)

In [11]:
aircraft_df = DataFrame(XLSX.readtable("INSTANCES/OAMRP-Data/25-aircraft/658FL_25A.xlsx", "Aircrafts"))

aircraft_airport = Dict()
for row in eachrow(aircraft_df)
    aircraft_airport[row.TAIL_NUMBER] = row.INIT_AIRPORT
end

XLSX.openxlsx("INSTANCES/OAMRP-Data/25-aircraft/Cases_21-30.xlsx", mode="w") do xf
    for i in 21:30
        nbr_ac_mtn = 12
        result_df = generate_remaining_time(aircraft_airport, nbr_ac_mtn)
        if i == 21
            sheet = xf[1]
            XLSX.rename!(sheet, "Sheet21")
        else
            sheet = XLSX.addsheet!(xf, "Sheet$i")
        end
        XLSX.writetable!(sheet, result_df)
    end
end

Avions sélectionnés : Any["N285AK", "N263AK", "N320AS", "N589AS", "N277AK", "N440AS", "N558AS", "N493AS", "N280AK", "N290AK", "N588AS", "N306AS"]
Avions sélectionnés : Any["N253AK", "N519AS", "N531AS", "N294AK", "N307AS", "N587AS", "N319AS", "N315AS", "N306AS", "N277AK", "N320AS", "N290AK"]
Avions sélectionnés : Any["N307AS", "N558AS", "N294AK", "N588AS", "N589AS", "N320AS", "N526AS", "N305AS", "N319AS", "N493AS", "N297AK", "N440AS"]
Avions sélectionnés : Any["N285AK", "N297AK", "N440AS", "N587AS", "N319AS", "N263AK", "N589AS", "N607AS", "N280AK", "N315AS", "N305AS", "N320AS"]
Avions sélectionnés : Any["N253AK", "N531AS", "N290AK", "N587AS", "N263AK", "N493AS", "N588AS", "N589AS", "N320AS", "N526AS", "N319AS", "N277AK"]
Avions sélectionnés : Any["N440AS", "N263AK", "N588AS", "N297AK", "N320AS", "N558AS", "N294AK", "N285AK", "N306AS", "N305AS", "N277AK", "N589AS"]
Avions sélectionnés : Any["N307AS", "N588AS", "N320AS", "N519AS", "N305AS", "N520AS", "N285AK", "N306AS", "N558AS", "N587AS"

In [13]:
data = DataFrame(XLSX.readtable("INSTANCES/OAMRP-Data/25-aircraft/Cases_21-30.xlsx", "Sheet21"))


Row,Aircraft,Airport,RemainingTime
,Any,Any,Any
1,N493AS,SAN,263
2,N285AK,SEA,306
3,N320AS,SJC,765
4,N440AS,JFK,820
5,N277AK,SEA,840
6,N280AK,SFO,933
7,N588AS,ORD,943
8,N589AS,SEA,1539
9,N290AK,MCO,1642


In [14]:
function computation(df_flights)
    tail_numbers = unique(df_flights.TAIL_NUMBER)
    aircraft_day_df = combine(groupby(df_flights, [:TAIL_NUMBER, :DAY]),
        :AIR_TIME => sum => :FLYING_TIME,
        :AIR_TIME => length => :TAKEOFF
    )

    sort!(aircraft_day_df, [:TAIL_NUMBER, :DAY])
    nbr_ac = length(tail_numbers)
    fh_ac_day = round(Int, sum(aircraft_day_df.FLYING_TIME)/nbr_ac/7)
    tk_ac_day = round(Int, sum(aircraft_day.TAKEOFF)/nbr_ac/7)
    fh_tk = round(sum(Int, aircraft_day.FLYING_TIME)/sum(aircraft_day.TAKEOFF))

    return (fh_ac_day = fh_ac_day, tk_ac_day = tk_ac_day, fh_tk = fh_tk)
end 

function computation_new(df_flights, nbr_ac)
    # Agrégation par jour pour obtenir les totaux quotidiens
    daily_stats = combine(groupby(df_flights, :DAY),
        :AIR_TIME => sum => :FLYING_TIME,
        :AIR_TIME => length => :TAKEOFF
    )
    
    # Calcul des moyennes par avion et par jour
    total_flying_time = sum(daily_stats.FLYING_TIME)
    total_takeoffs = sum(daily_stats.TAKEOFF)
    nbr_days = length(unique(df_flights.DAY))
    
    fh_ac_day = round(Int, total_flying_time / nbr_ac / nbr_days)
    tk_ac_day = round(Int, total_takeoffs / nbr_ac / nbr_days)
    fh_tk = round(Int, total_flying_time / total_takeoffs)
    
    return (fh_ac_day = fh_ac_day, tk_ac_day = tk_ac_day, fh_tk = fh_tk)
end

function process_xlsx_files_timer(folder_path)
    # Obtenir tous les fichiers .xlsx
    xlsx_files = filter(f -> endswith(f, ".xlsx"), readdir(folder_path))
    
    # Temps actuel
    current_time = time()
    
    # Filtrer les fichiers des 2 dernières minutes (120 secondes)
    recent_files = filter(xlsx_files) do filename
        filepath = joinpath(folder_path, filename)
        file_mtime = mtime(filepath)  # Temps de modification
        (current_time - file_mtime) <= 1200 # 120 secondes = 2 minutes
    end
    
    for filename in recent_files
        filepath = joinpath(folder_path, filename)
        println("Traitement de: $filename")
        # Traiter le fichier
        # wb = XLSX.readxlsx(filepath)
        # ... votre code de traitement ...
    end
    
    return recent_files
end


process_xlsx_files_timer (generic function with 1 method)

In [20]:
# Charger la feuille
datas = []
inst = "INSTANCES/OAMRP-Data/25-aircraft/658FL_25A"
file = inst*".xlsx"
for i in 11:20
    data = DataFrame(XLSX.readtable("INSTANCES/OAMRP-Data/25-aircraft/Cases_11-20.xlsx", "Sheet$i"))
    push!(datas, data)
    df_flight = DataFrame(XLSX.readtable(file, "Data"))
    df_param = DataFrame(XLSX.readtable(file, "Parameters"))
    df_mstations = DataFrame(XLSX.readtable(file, "M_stations"))
    df_aircrafts = DataFrame(XLSX.readtable(file, "Aircrafts"))
    df_aircrafts.INIT_AIRPORT = data.Airport
    nbr_ac = nrow(df_aircrafts)

    result = computation_new(df_flight,nbr_ac)
    df_param.FH_DAY = [result.fh_ac_day]
    df_param.FH_TK = [result.fh_tk]
    df_param.TK_DAY = [result.tk_ac_day]
    df_param.T = [floor(df_param.F[1]/result.fh_tk)]
    df_param.D = [floor(df_param.F[1]/result.fh_ac_day)]

    for (i, ms) in enumerate(df_mstations.MTN_STATIONS)
        new_values = if ms == "SEA"
            [rand() < 0.1 ? 1 : rand(2:5) for _ in 1:7]
        else
            [rand() < 0.1 ? 0 : 1 for _ in 1:7]
        end
        df_mstations[i, 2:end] = new_values
    end
    df_aircrafts.TAIL_NUMBER = data.Aircraft
    for j in 1:nbr_ac
        df_aircrafts.INIT_AIRPORT[j] = data.Airport[j]
        if occursin(">", string(data[j, "RemainingTime"]))
            df_aircrafts.INIT_FLYING_TIME[j] = 0
            df_aircrafts.INIT_TAKEOFF[j] = 0
            df_aircrafts.INIT_FLYING_DAY[j] = 1
        else
            df_aircrafts.INIT_FLYING_TIME[j] = 6000 - data[j, "RemainingTime"]
            df_aircrafts.INIT_TAKEOFF[j] = floor((6000- data[j, "RemainingTime"])/result.fh_tk)
            df_aircrafts.INIT_FLYING_DAY[j] = floor((6000- data[j, "RemainingTime"])/result.fh_ac_day)
        end
    end
    
    filename = inst*"_"*string(i)*".xlsx"
    XLSX.openxlsx(filename, mode = "w") do xf
        # Supprimer la feuille par défaut "Sheet1"
        XLSX.rename!(xf["Sheet1"], "Data")
        data_sheet = xf["Data"]

        #data_sheet = XLSX.addsheet!(xf, "Data")
        parameters_sheet = XLSX.addsheet!(xf, "Parameters")
        mtn_stations_sheet = XLSX.addsheet!(xf, "M_stations")
        aircrafts_sheet = XLSX.addsheet!(xf, "Aircrafts")

        XLSX.writetable!(data_sheet, Tables.columntable(df_flight); write_columnnames = true)
        XLSX.writetable!(parameters_sheet, Tables.columntable(df_param); write_columnnames = true)
        XLSX.writetable!(mtn_stations_sheet, Tables.columntable(df_mstations); write_columnnames = true)
        XLSX.writetable!(aircrafts_sheet, Tables.columntable(df_aircrafts); write_columnnames = true)
        println("Fichier xlsx créé")
    end
end

Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé


In [21]:
for ac_critique in [12]
    folder_path ="INSTANCES/instances_new_xlsx/A_MTN_"*string(ac_critique)*"/"  # ou le chemin vers votre dossier
    xlsx_files = process_xlsx_files_timer(folder_path)

    for file in xlsx_files
        df_flight = DataFrame(XLSX.readtable(folder_path*file, "Data"))
        df_param = DataFrame(XLSX.readtable(folder_path*file, "Parameters"))
        df_mstations = DataFrame(XLSX.readtable(folder_path*file, "M_stations"))
        df_aircrafts = DataFrame(XLSX.readtable(folder_path*file, "Aircrafts"))
        #df_inventory = DataFrame(XLSX.readtable(folder_path*file*".xlsx", "Inventory"))

        O_airport = unique(df_flight.ORIGIN_AIRPORT)
        D_airport = unique(df_flight.DESTINATION_AIRPORT)
        airports = unique(vcat(O_airport, D_airport))
        aircrafts = unique(df_aircrafts.TAIL_NUMBER)
        nbr_flights = nrow(df_flight)
        nbr_airports = length(airports)
        nbr_aircrafts = length(aircrafts)

        mtn_stations = df_mstations.MTN_STATIONS
        initial_airport = Dict(row.TAIL_NUMBER => row.INIT_AIRPORT for row in eachrow(df_aircrafts))
        nbr_mstations = length(mtn_stations)
        initial_flying_time = Dict(row.TAIL_NUMBER => row.INIT_FLYING_TIME for row in eachrow(df_aircrafts))
        initial_takeoff = Dict(row.TAIL_NUMBER => row.INIT_TAKEOFF for row in eachrow(df_aircrafts))
        initial_flying_day = Dict(row.TAIL_NUMBER => row.INIT_FLYING_DAY for row in eachrow(df_aircrafts))
        #ms_capacity = Dict(row.MTN_STATIONS => [] for row in eachrow(df_mstations))
        ms_capacity = Dict(row.MTN_STATIONS => collect(row[2:end]) for row in eachrow(df_mstations))
        turn_around_time = df_param.TRT[1]
        flying_time_max = df_param.F[1]
        takeoff_max = df_param.T[1]
        flying_day_max = df_param.D[1]
        mtn_time = df_param.MT[1]
        nbr_TP = df_param.NBR_TP[1]
        fh_tk = df_param.FH_TK[1]
        fh_day = df_param.FH_DAY[1]
        tk_day = df_param.TK_DAY[1]

        #= 
        exp_part = df_inventory.EXP_PART
        nbr_exp_part = length(exp_part)
        init_level_ep = Dict(row.EXP_PART => row.INIT_LEVEL for row in eachrow(df_inventory))
        rate_ep = Dict(row.EXP_PART => row.RATE for row in eachrow(df_inventory))
        =#
        df_copy = deepcopy(df_flight)
        #df_copy = time_to_minutes(df_copy)
        temp1 = maximum(df_flight.DAY)*1440
        temp2 = maximum(df_copy.ARRIVAL_TIME)
        end_horizon_time = maximum([temp1, temp2])

        flight_legs = []
        for row in eachrow(df_copy)
            push!(flight_legs, (String(row.ORIGIN_AIRPORT), String(row.DESTINATION_AIRPORT), Int(row.DEPARTURE_TIME), Int(row.ARRIVAL_TIME), Int(row.AIR_TIME)))
        end

        instance_data = Dict(
            "number_of_flight_legs" => nbr_flights, "number_of_airports" => nbr_airports,
            "number_of_maintenance_stations" => nbr_mstations, "number_of_aircrafts" => nbr_aircrafts,
            "aircrafts" => aircrafts, "maximum_flying_time" => flying_time_max,
            "maximum_takeoff" => takeoff_max, "maximum_flying_day" => flying_day_max, "airports" => airports,
            "maintenance_stations" => mtn_stations, "initial_flying_time" => initial_flying_time,
            "initial_takeoff" => initial_takeoff, "initial_flying_day" => initial_flying_day,
            "mtn_station_capacity" => ms_capacity, "initial_airport_aircraft" => initial_airport, 
            "flight_legs" => flight_legs, "turn_around_time" => turn_around_time, 
            "maintenance_time" => mtn_time, "end_horizon_time" => end_horizon_time,
            "nbr_TP" => nbr_TP, "fh_tk" => fh_tk, "fh_day" => fh_day, "tk_day" => tk_day
            #= "exp_part" => exp_part,
            "number_of_exp_part" => nbr_exp_part,
            "init_level_ep" => init_level_ep,
            "rate_ep" => rate_ep =#
        )

        # S'assurer que le dossier existe avant d'écrire le fichier JSON
        json_filepath = "INSTANCES/instances_new_json/A_MTN_"*string(ac_critique)*"/" * string(splitext(file)[1]) * ".json"
        dirpath = dirname(json_filepath)
        if !isdir(dirpath)
            mkpath(dirpath)
        end

        # Sauvegarde au format JSON dans un fichier
        open(json_filepath, "w") do f
            JSON.print(f, instance_data;)
            println("Fichier json créé")
        end
    end
end

Traitement de: 658FL_25A_11.xlsx
Traitement de: 658FL_25A_12.xlsx
Traitement de: 658FL_25A_13.xlsx
Traitement de: 658FL_25A_14.xlsx
Traitement de: 658FL_25A_15.xlsx
Traitement de: 658FL_25A_16.xlsx
Traitement de: 658FL_25A_17.xlsx
Traitement de: 658FL_25A_18.xlsx
Traitement de: 658FL_25A_19.xlsx
Traitement de: 658FL_25A_20.xlsx
Fichier json créé
Fichier json créé
Fichier json créé
Fichier json créé
Fichier json créé
Fichier json créé
Fichier json créé
Fichier json créé
Fichier json créé
Fichier json créé
